In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

In [ ]:
# ⬇️ Параметры подключения к PostgreSQL 
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shops" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 

shops_df = (spark.read
            .format("jdbc")
            .option("url", jdbc_url)
            .option("user", db_user)
            .option("password", db_password)
            .option("dbtable", table_name)
            .option("fetchsize", 1000)
            .option("driver", "org.postgresql.Driver")
            .load()
            )

shops_df.show()

In [ ]:
# ⬇️ Параметры подключения к PostgreSQL 
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shop_timezone" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 

shop_timezone_df = ( spark.read
				.format("jdbc")
				.option("url", jdbc_url)
				.option("user", db_user)
				.option("password", db_password)
				.option("dbtable", table_name) 
				.option("fetchsize", 1000)
				.option("driver", "org.postgresql.Driver")
                .load()
)

shop_timezone_df.show()

In [ ]:
shops_df.createOrReplaceTempView("shops") 
shop_timezone_df.createOrReplaceTempView("shop_timezone")

# Выполняем SQL запрос для трансформации 
final_df = spark.sql("""        
                        WITH sz AS (
                        SELECT  TRY_CAST(plant AS INT) AS id, 
                                coalesce(time_zone, '') AS time_zone
                        FROM shop_timezone 
                        WHERE coalesce(TRY_CAST(plant AS INT), 0) > 0
                        ), 
                        s AS ( 
                        SELECT  TRY_CAST(st_id AS INT) AS st_id, 
                                shop_name 
                        FROM shops
                        ), 
                        result AS (
                        SELECT  *, 
                                ROW_NUMBER() over (PARTITION BY st_id ORDER BY time_zone DESC) AS rn
                        FROM s 
                        JOIN sz 
                                ON s.st_id = sz.id ) 
                        SELECT  st_id, 
                        shop_name, 
                        TRY_CAST(CASE WHEN time_zone = '' THEN 3 ELSE substr(time_zone, 4) END AS INT) AS tz_code 
                        FROM result 
                        WHERE rn = 1 
                """
)

final_df.show()

In [ ]:
spark.stop()